# Lab 11 · Đo chất lượng đầu ra của LLM

**Lập trình xử lý dữ liệu · 2627-1 · Giờ thực hành · Bài 11**

Chọn **Save a copy in Drive** trước khi sửa.

Lab đi theo mục 4–5 của bài giảng và **không cần khoá API**. Bạn sẽ chọn mẫu đánh giá và ước tính chi phí,
viết schema, rồi dùng nhãn tay và quy tắc hậu kiểm để đo một bộ đầu ra LLM mô phỏng có lỗi cài sẵn.

**Tài liệu:** [Gemini API: Structured output](https://ai.google.dev/gemini-api/docs/structured-output),
[Pydantic: Models](https://docs.pydantic.dev/latest/concepts/models/).
Lab dùng thêm `str.len`, `sample` (Bài 4–5) và cách thay thẻ `<br/>` (Bài 7).

## Cách làm việc trong bài lab

- Điền các ô `TODO` và phần giải thích.
  `assert` kiểm tra một số điều kiện, không thay thế việc xem các trường hợp sai.
- Với các phần có hướng dẫn, tự viết trước khi nhờ AI.
  Các bài kiểm tra trên lớp (🚫 đóng) đánh giá các kỹ năng này.
- Bài tự làm ✅ mở cho phép dùng AI, kèm khai báo và cách kiểm chứng.
- Khi cần hỗ trợ, trao đổi với giảng viên thực hành về yêu cầu hoặc trường hợp chưa hiểu.

## Mục tiêu

1. Chọn mẫu đánh giá gửi cho LLM và ước tính chi phí trước khi gọi.
2. Viết schema Pydantic có danh sách giá trị được phép; kiểm tra schema chặn được những đầu ra nào.
3. Đo tỷ lệ đúng của đầu ra LLM trên nhãn tay và phân loại lỗi.
4. Viết quy tắc hậu kiểm và so với kết quả đo bằng nhãn tay.

## Chuẩn bị

Dữ liệu thật là bảng đánh giá của Santiago, mốc chụp **29/06/2026**, bản đầy đủ `data/reviews.csv.gz`
(khoảng 59 MB): mỗi hàng là một đánh giá; `id` là mã đánh giá, `comments` là nội dung.
Ô dưới chỉ đọc hai cột này, bỏ đánh giá thiếu nội dung và thay thẻ xuống dòng `<br/>` (cùng ký tự `\r` đứng trước, nếu có) bằng dấu cách như ở bài giảng.

Dùng file cục bộ nếu có; nếu chưa có thì đọc đúng mốc chụp từ nguồn.
Chạy hai ô dưới trước. Nếu phiên Colab đã nạp pandas phiên bản khác, chọn
**Restart session and run all** sau khi cài.

In [ ]:
%pip -q install pandas==3.0.3

In [ ]:
from pathlib import Path
from typing import Literal

import pandas as pd
from pydantic import BaseModel, ValidationError

URL = ("https://data.insideairbnb.com/chile/rm/santiago/"
       "2026-06-29/data/reviews.csv.gz")
tep = Path("data/reviews_santiago_full_2026-06-29.csv.gz")
rv = pd.read_csv(tep if tep.exists() else URL, usecols=["id", "comments"])
rv = rv.dropna(subset=["comments"])
rv["comments"] = rv["comments"].str.replace(r"\r?<br/>", " ", regex=True)
print("pandas:", pd.__version__)
print("Số đánh giá có nội dung:", len(rv))

## Phần 1 · Chọn mẫu và ước tính chi phí

Không gửi cả bảng cho mô hình: gói miễn phí có hạn mức, gói trả phí tính tiền theo token.
Lab này lấy mẫu 100 đánh giá để gán nhãn tay.

Ngưỡng 50 ký tự là quyết định chọn mẫu của bài này: ưu tiên đánh giá đủ dài để có nội dung gán khía cạnh.
Ngưỡng này loại 154.357/690.076 đánh giá có nội dung (22,4%), nên kết quả đo trên mẫu chỉ nói về
các đánh giá từ 50 ký tự trở lên.

**a)** Tạo `du_dai` gồm các đánh giá có `comments` dài từ 50 ký tự; `mau` gồm 100 đánh giá
lấy ngẫu nhiên từ `du_dai` với `random_state=42`.

In [ ]:
# TODO: tạo du_dai và mau.
du_dai = None
mau = None

In [ ]:
assert du_dai is not None and mau is not None, "Tạo du_dai và mau."
assert len(du_dai) == 535_719, "Giữ đánh giá có độ dài chuỗi từ 50 ký tự trở lên (>= 50)."
assert du_dai["comments"].str.len().min() >= 50, "du_dai còn đánh giá ngắn hơn 50 ký tự."
assert len(mau) == 100, "Mẫu có đúng 100 đánh giá."
assert mau.index.isin(du_dai.index).all(), "Lấy mẫu từ du_dai, không lấy từ rv."
assert mau.equals(du_dai.sample(100, random_state=42)), "Dùng random_state=42 để mẫu tái lập được."
print(f"Chọn 100/{len(du_dai):,} đánh giá đủ dài.")

**b)** Token là đơn vị văn bản mô hình xử lý và tính tiền. Ước tính số token theo quy tắc kinh nghiệm:
khoảng 4 ký tự một token. Tính `tong_token`: lấy `len // 4` của từng đánh giá trong mẫu rồi cộng lại.
Tính `chi_phi` (USD) cho phần dữ liệu này với giá đầu vào của
`gemini-3.5-flash-lite`: **0,30 USD mỗi 1 triệu token** (trang Pricing, 19/10/2026).

In [ ]:
# TODO: tính tong_token và chi_phi.
tong_token = None
chi_phi = None

In [ ]:
assert tong_token is not None and chi_phi is not None, "Tính tong_token và chi_phi."
assert tong_token == int((mau["comments"].str.len() // 4).sum()), "Lấy len // 4 của từng đánh giá trong mau rồi cộng lại."
assert abs(chi_phi - tong_token * 0.30 / 1_000_000) < 1e-12, "Giá 0,30 USD cho mỗi 1.000.000 token."
print(f"~{tong_token:,} token dữ liệu ≈ {chi_phi:.5f} USD (chưa tính prompt và token ra).")

**Nhận xét của bạn:** Ở bài giảng, mỗi lời gọi tốn khoảng 81 token vào và 49 token ra dù đánh giá minh hoạ rất ngắn.
Phần lớn token vào đến từ đâu? Vì sao chi phí token ra lại lớn hơn chi phí token vào?
Ước lượng 4 ký tự/token có hợp với đánh giá tiếng Tây Ban Nha không, và bạn kiểm tra điều này bằng cách nào? …

## Phần 2 · Schema là bước kiểm tra đầu tiên

Viết lớp `ReviewInfo` gồm 4 trường như bài giảng:

- `sentiment`: chỉ nhận `"positive"`, `"mixed"`, `"negative"`;
- `aspects_positive`, `aspects_negative`: danh sách, mỗi phần tử thuộc `Aspect`;
- `language`: chuỗi.

In [ ]:
Aspect = Literal["location", "cleanliness", "host", "noise", "amenities", "value"]

# TODO: viết class ReviewInfo(BaseModel).
class ReviewInfo(BaseModel):
    pass

In [ ]:
def bi_chan(tho):
    try:
        ReviewInfo.model_validate_json(tho)
    except ValidationError:
        return True
    return False

assert set(ReviewInfo.model_fields) == {"sentiment", "aspects_positive", "aspects_negative", "language"}, (
    "ReviewInfo cần đúng 4 trường: sentiment, aspects_positive, aspects_negative, language.")
dung = ('{"sentiment": "mixed", "aspects_positive": ["location"], '
        '"aspects_negative": ["noise"], "language": "en"}')
assert not bi_chan(dung), "Đầu ra đúng schema phải được nhận."
assert ReviewInfo.model_validate_json(dung).aspects_negative == ["noise"], "Giữ tên trường như đề."
assert bi_chan(dung.replace('"mixed"', '"happy"')), "sentiment chỉ nhận positive, mixed, negative."
assert bi_chan(dung.replace('["noise"]', '["wifi_speed"]')), "Mỗi khía cạnh bị chê phải thuộc Aspect."
assert bi_chan(dung.replace('["location"]', '["wifi_speed"]')), "Mỗi khía cạnh được khen phải thuộc Aspect."
assert bi_chan(dung.replace('["noise"]', '"noise"')), "aspects_negative là danh sách, không phải chuỗi."
assert bi_chan(dung.replace(', "language": "en"', '')), "Thiếu language phải bị chặn."
print("Schema chặn đúng 5 kiểu đầu ra sai.")

## Phần 3 · Đo đầu ra LLM bằng nhãn tay

Ô dưới lấy **10 đánh giá thật** của Santiago theo mã đánh giá (gộp khoảng trắng thừa), kèm nhãn tay `GOLD`
và **10 đầu ra LLM mô phỏng** `LLM_OUT`. Đầu ra mô phỏng có lỗi cài sẵn; không sửa `LLM_OUT`.
Quy ước gán nhãn của bài lab: có cả khen lẫn chê → `mixed`; đánh giá không có nội dung → `mixed`,
không có khía cạnh, ngôn ngữ `"und"`.

In [ ]:
MA_DANH_GIA = {
    1: 659371629434460707, 2: 593056807, 3: 89504981,
    4: 1653700011604781333, 5: 59139461, 6: 442072129,
    7: 1315303353459720088, 8: 718026519374709265,
    9: 1637796258939429124, 10: 637696848251764864,
}
noi_dung = rv.set_index("id")["comments"]
REVIEWS = {k: " ".join(noi_dung[ma].split()) for k, ma in MA_DANH_GIA.items()}

GOLD = {
    1: {"sentiment": "mixed",    "language": "en"},
    2: {"sentiment": "negative", "language": "en"},
    3: {"sentiment": "positive", "language": "es"},
    4: {"sentiment": "mixed",    "language": "es"},
    5: {"sentiment": "mixed",    "language": "en"},
    6: {"sentiment": "positive", "language": "en"},
    7: {"sentiment": "positive", "language": "es"},
    8: {"sentiment": "mixed",    "language": "und"},
    9: {"sentiment": "mixed",    "language": "en"},
    10: {"sentiment": "mixed",   "language": "en"},
}

LLM_OUT = {
    1: '{"sentiment": "mixed", "aspects_positive": ["cleanliness", "location"], "aspects_negative": ["noise"], "language": "en"}',
    2: '{"sentiment": "negative", "aspects_positive": ["host"], "aspects_negative": ["photos", "cleanliness"], "language": "en"}',
    3: '{"sentiment": "positive", "aspects_positive": ["host", "location"], "aspects_negative": [], "language": "es"}',
    4: '{"sentiment": "positive", "aspects_positive": ["location"], "aspects_negative": [], "language": "es"}',
    5: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities", "host"]}',
    6: '{"sentiment": "positive", "aspects_positive": ["amenities", "location"], "aspects_negative": [], "language": "en"}',
    7: '{"sentiment": "positive", "aspects_positive": ["location", "noise"], "aspects_negative": [], "language": "es"}',
    8: '{"sentiment": "mixed", "aspects_positive": [], "aspects_negative": [], "language": "und"}',
    9: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["noise", "cleanliness"], "language": "en"}',
    10: '{"sentiment": "negative", "aspects_positive": ["location"], "aspects_negative": ["amenities"], "language": "en"}',
}
for k, t in REVIEWS.items():
    print(f"[{k}] {t[:100]}")

**a)** Kiểm tra từng chuỗi trong `LLM_OUT` bằng `ReviewInfo`. Đầu ra hợp lệ đưa vào dict `hop_le`
(mã → đối tượng `ReviewInfo`); mã của đầu ra bị chặn đưa vào list `loi_schema`.

In [ ]:
# TODO: tạo hop_le và loi_schema.
hop_le = {}
loi_schema = []

In [ ]:
assert sorted(loi_schema) == [2, 5], "Hai đầu ra sai schema; xem lại cách bắt ValidationError."
assert sorted(hop_le) == [1, 3, 4, 6, 7, 8, 9, 10], "hop_le gồm 8 đầu ra còn lại."
assert all(isinstance(r, ReviewInfo) for r in hop_le.values()), "Giá trị trong hop_le là đối tượng ReviewInfo."
for ma in loi_schema:
    try:
        ReviewInfo.model_validate_json(LLM_OUT[ma])
    except ValidationError as loi:
        e = loi.errors()[0]
        print(f"[{ma}] {e['type']} tại {e['loc']}")

Đầu ra 2 có nhãn `photos` ngoài danh sách; đầu ra 5 thiếu trường `language`. Cả hai bị chặn trước khi vào bảng.
Trong pipeline thật, đầu ra bị chặn được ghi lại để gọi lại hoặc kiểm tra bằng tay.

**b)** Trên 8 đầu ra hợp lệ, đếm `so_dung` là số đánh giá có `sentiment` trùng `GOLD`, và `ti_le` là tỷ lệ đúng
(dạng số từ 0 đến 1).

In [ ]:
# TODO: tính so_dung và ti_le.
so_dung = None
ti_le = None

In [ ]:
assert so_dung is not None and ti_le is not None, "Tính so_dung và ti_le."
assert so_dung == 5, "Đếm trên 8 đầu ra hợp lệ, so sentiment với GOLD."
assert abs(ti_le - 5 / 8) < 1e-9, "ti_le = so_dung / số đầu ra hợp lệ, dạng số từ 0 đến 1."
sai = sorted(ma for ma, r in hop_le.items() if r.sentiment != GOLD[ma]["sentiment"])
print(f"Tỷ lệ đúng: {ti_le:.1%}; sai ở {sai}")
for ma in sai:
    print(f"[{ma}] nhãn tay={GOLD[ma]['sentiment']}, LLM={hop_le[ma].sentiment} | {REVIEWS[ma]}")

**c)** Đọc ba đánh giá bị gán sai ở trên rồi xếp mỗi đánh giá vào một kiểu lỗi. Mỗi kiểu dùng đúng một lần.

- `"A"`: **bỏ sót ý chê** — lời khen dài che mất một nhận xét chê, mô hình gán `positive`;
- `"B"`: **ranh giới nhãn** — các ý khen, chê đều nói thẳng, không có cụm phủ định bị đọc sai;
  người gán chọn `mixed`, mô hình chọn `negative`;
- `"C"`: **hiểu sai phủ định** — một cụm phủ định mang nghĩa khen (vd. “not a bad place”) bị đọc thành chê.

In [ ]:
# TODO: điền kiểu lỗi "A", "B" hoặc "C" cho từng mã.
kieu_loi = {4: None, 9: None, 10: None}

In [ ]:
assert set(kieu_loi) == {4, 9, 10}, "Giữ đúng ba mã 4, 9, 10."
assert None not in kieu_loi.values(), "Điền 'A', 'B' hoặc 'C' cho từng mã."
assert sorted(map(str, kieu_loi.values())) == ["A", "B", "C"], "Mỗi kiểu A, B, C dùng đúng một lần."
assert kieu_loi[4] == "A", "Đánh giá 4: tìm câu chê nằm giữa các câu khen."
assert kieu_loi[10] == "C", "Đánh giá 10: đọc lại cụm có chữ not ở cuối câu."
assert kieu_loi[9] == "B", "Đánh giá 9: có cả 'Nice place' lẫn hai ý chê."
print("Phân loại khớp với hướng dẫn gán nhãn của bài lab.")

Đánh giá 9 cho thấy người gán nhãn cũng có thể không đồng ý với nhau. Hướng dẫn gán nhãn đã thống nhất
là căn cứ để chấm, và phải được áp dụng nhất quán.

**d)** Hậu kiểm: một đầu ra tự mâu thuẫn khi `sentiment` là `"positive"` mà `aspects_negative` không rỗng,
hoặc `"negative"` mà `aspects_positive` không rỗng. Viết hàm `tu_mau_thuan(r)` nhận một đối tượng `ReviewInfo`
và trả `True` khi đầu ra tự mâu thuẫn.

In [ ]:
# TODO: viết hàm tu_mau_thuan.
def tu_mau_thuan(r):
    return None

In [ ]:
def tao(sentiment, khen, che):
    return ReviewInfo(sentiment=sentiment, aspects_positive=khen, aspects_negative=che, language="en")

assert tu_mau_thuan(tao("positive", ["host"], [])) is not None, "Hàm cần trả True hoặc False."
assert tu_mau_thuan(tao("positive", ["host"], ["noise"])), "positive mà có ý chê là mâu thuẫn."
assert tu_mau_thuan(tao("negative", ["host"], ["noise"])), "negative mà có ý khen là mâu thuẫn."
assert not tu_mau_thuan(tao("mixed", ["host"], ["noise"])), "mixed có cả khen lẫn chê là hợp lý."
assert not tu_mau_thuan(tao("positive", ["host"], [])), "positive chỉ có ý khen là hợp lý."
mau_thuan = sorted(ma for ma, r in hop_le.items() if tu_mau_thuan(r))
sai = sorted(ma for ma, r in hop_le.items() if r.sentiment != GOLD[ma]["sentiment"])
print(f"Hậu kiểm gắn cờ {mau_thuan}; các ca gán sai ở câu b) là {sai}.")

**Nhận xét của bạn:** Quy tắc hậu kiểm không cần nhãn tay nhưng bắt được những ca nào trong ba ca gán sai,
và bỏ sót ca nào? Đánh giá 7 có `aspects_positive` chứa `noise` vì khách khen “sin ruido” (yên tĩnh):
đây có phải lỗi không, và schema của bạn cần quy ước gì để gán nhãn nhất quán? …

## Phần 4 · Bài tự làm ✅ mở (làm sớm tại lớp hoặc làm tại nhà)

### Tự làm 1 · Tỷ lệ đúng trên cả 10 đánh giá

Ở câu 3b, tỷ lệ đúng tính trên 8 đầu ra hợp lệ. Tính `ti_le_10`: tỷ lệ đúng trên cả 10 đánh giá,
coi đầu ra bị chặn là sai. Báo cáo con số nào khi so LLM với phương pháp đối chứng, và vì sao?

In [ ]:
# TODO: tính ti_le_10.
ti_le_10 = None

In [ ]:
assert ti_le_10 is not None, "Tính ti_le_10."
assert abs(ti_le_10 - 0.5) < 1e-9, "Mẫu số là 10; đầu ra bị chặn tính là sai."
print(f"Trên 10 đánh giá: {ti_le_10:.0%}; trên 8 đầu ra hợp lệ: {ti_le:.1%}")

**Nhận xét của bạn:** …

### Tự làm 2 · Chạy thật với khoá API của bạn

Chép từ notebook bài giảng các ô: khoá API (`CO_KHOA`), tạo `client` và `MODEL`, `PROMPT_TEMPLATE`, `goi_llm`,
`trich_xuat`; thêm `import json, time` và cài `google-genai>=2.19`. Dùng `ReviewInfo` của lab này.
**Không chép ô `KET_QUA_LUU_SAN` của bài giảng**: bắt đầu với `LLM_CACHE = {}` và lưu ra file riêng, ví dụ `lab11_cache.json`.
Gọi Gemini cho 10 đánh giá trong `REVIEWS`, rồi: (1) kiểm tra schema như câu 3a; (2) đo tỷ lệ đúng với `GOLD`
như câu 3b; (3) so với đầu ra mô phỏng. Mô hình thật có đọc sai cụm phủ định ở đánh giá 10 không?

In [ ]:
# Viết bài tự làm 2 ở đây (cần GEMINI_API_KEY trong Colab Secrets).

**Khai báo AI (nếu dùng):** công cụ, yêu cầu chính, một kết quả bạn đã tự kiểm tra. …

## Hỗ trợ bài tập lớn (làm theo nhóm)

Trọng tâm tuần này: **thiết lập hợp phần LLM kiểm chứng được và chạy lại được.**

1. ☐ Khoá API Gemini nằm trong **Colab Secrets** hoặc biến môi trường, chưa từng xuất hiện trong code hay commit nào
   (soát bằng `git log -p | grep`).
2. ☐ **Mẫu đánh giá** để gán nhãn đã chốt: cách lọc theo độ dài và thời gian, `random_state`, đủ số lượng đề yêu cầu,
   tỷ lệ các ngôn ngữ chính.
3. ☐ **Schema và danh mục khía cạnh** bản nháp đã viết; danh mục hợp với câu hỏi phân tích của nhóm.
4. ☐ **Hướng dẫn gán nhãn** 5–10 dòng đã viết chung (đánh giá rỗng, `mixed`, nhiều ngôn ngữ); mỗi đánh giá
   do hai người gán độc lập, rồi bàn lại các chỗ bất đồng.
5. ☐ Phương pháp đối chứng không dùng LLM (từ khoá, Bài 7) chạy được trên đúng mẫu đó.

Nhóm xong sớm: chạy thử 5 lời gọi Gemini với schema của nhóm và kiểm tra đầu ra có đúng cấu trúc không.

## Tóm tắt bài lab

- Chọn mẫu có quy tắc rõ và `random_state`; kết quả đo chỉ nói về phạm vi của mẫu.
- Ước tính chi phí trước khi gọi: token vào gồm cả prompt hướng dẫn; token ra thường đắt hơn.
- Schema chặn nhãn lạ, sai kiểu và trường thiếu, nhưng không chặn nhãn hợp lệ mà sai nội dung.
- Tỷ lệ đúng phải nêu rõ mẫu số; đầu ra bị chặn vẫn tính là một lần LLM không cho kết quả dùng được.
- Hậu kiểm chạy trên mọi đầu ra mới nhưng chỉ bắt một số kiểu lỗi; nhãn tay đo chất lượng tổng thể.

Chạy **Restart session and run all**, rồi kiểm tra: mọi ô code đều có kết quả; các ô
nhận xét và khai báo AI (nếu dùng) đã điền đủ.

**Đọc thêm:** [Gemini API: Structured output](https://ai.google.dev/gemini-api/docs/structured-output),
[Pydantic: Models](https://docs.pydantic.dev/latest/concepts/models/).

**Bài sau:** trực quan hoá dữ liệu cơ bản.